# Calculating LCOH

Here we calculate the Levelised Cost of Hydrogen (LCOH) for electrolytic hydrogen.

## Prepare notebook.

In [1]:
import numpy as np
import pandas as pd
import plotly

pd.options.plotting.backend = "plotly"

from cet_units import Q
from posted import TEDF
from team.tools import calc_LCOX

## Obtain data

In [2]:
tech_ely = TEDF.load("Tech|Electrolysis").aggregate(
    subtech=["AEL", "PEM"],
    size="100 MW",
    append_references=True,
    agg=["source", "subtech"],
)
display(tech_ely)

,variable,value,unit
0,CAPEX,681.732210,EUR_2024
1,Input Capacity|Electricity,1.000000,kW
2,Input|Electricity,1.000000,kWh
3,Input|Water,0.178894,kg_H2O
4,OPEX Fixed,19.736287,EUR_2024/year
5,Output|Hydrogen,0.019627,kg_H2
6,Total Input Capacity|Electricity,52.500000,MW


## Choose assumptions

In [3]:
assumptions = pd.concat([
    pd.DataFrame().from_records([
        {"variable": "Price|Water", "value": 10.0, "unit": "EUR_2024/t_H2O"},
        {"variable": "OCF", "value": 50.0, "unit": "percent"},
    ]),
    pd.DataFrame({"price_case": ["low", "high"], "variable": "Price|Electricity", "value": [30.0, 60.0], "unit": "EUR_2024/MWh"}),
], ignore_index=True).team.sort_columns()
display(assumptions)

,price_case,variable,unit,value
0,NaN,Price|Water,EUR_2024/t_H2O,10.0
1,NaN,OCF,percent,50.0
2,low,Price|Electricity,EUR_2024/MWh,30.0
3,high,Price|Electricity,EUR_2024/MWh,60.0


## Calculate LCOH for different electricity prices

In [4]:
df = (
    tech_ely
    .team.perform(calc_LCOX, reference="Output|Hydrogen", using=assumptions, interest_rate=0.08, book_lifetime="20 years", only_new=True)
    .team.unit_to("EUR_2024/MWh_H2_LHV")
    .sort_values(by="price_case", key=lambda col: col.apply(["low", "high"].index))
)
display(df)

display(
    df
    .plot.bar(
        x="price_case",
        y="value",
        color="variable",
    )
    .update_layout(
        xaxis_title="Electricity price case",
        yaxis_title=f"LCOH  ( {df.unit.iloc[0]} )",
    )
)

,price_case,variable,unit,value
1,low,LCOX|Capital,EUR_2024/MWh_H2_LHV,24.231319
3,low,LCOX|OM Fixed,EUR_2024/MWh_H2_LHV,6.887446
7,low,LCOX|Input Cost|Water,EUR_2024/MWh_H2_LHV,2.734405
5,low,LCOX|Input Cost|Electricity,EUR_2024/MWh_H2_LHV,45.855154
2,high,LCOX|OM Fixed,EUR_2024/MWh_H2_LHV,6.887446
0,high,LCOX|Capital,EUR_2024/MWh_H2_LHV,24.231319
4,high,LCOX|Input Cost|Electricity,EUR_2024/MWh_H2_LHV,91.710307
6,high,LCOX|Input Cost|Water,EUR_2024/MWh_H2_LHV,2.734405


## Calculate LCOH for different capacity factors

In [5]:
assumptions = pd.concat([
    pd.DataFrame().from_records([
        {"variable": "Price|Electricity", "value": 50.0, "unit": "EUR_2024/MWh"},
        {"variable": "Price|Water", "value": 10.0, "unit": "EUR_2024/t_H2O"},
    ]),
    pd.DataFrame({"ocf": pd.Series(range(10, 100)), "variable": "OCF", "value": np.arange(10.0, 100.0), "unit": "percent"}),
])

In [6]:
df = (
    tech_ely
    .team.perform(calc_LCOX, reference="Output|Hydrogen", using=assumptions, interest_rate=0.08, book_lifetime="20 years", only_new=True)
    .team.unit_to("EUR_2024/kg_H2")
    .team.sum_over("variable")
)
display(df)

display(
    df
    .plot.line(
        x="ocf",
        y="value",
    )
    .update_layout(
        xaxis_title="Operational Capacity Factor  (%)",
        yaxis_title=f"LCOH  ( {df.unit.iloc[0]} )",
    )
)

,ocf,value,unit
0,10.0,7.825116,EUR_2024/kg_H2
1,11.0,7.353620,EUR_2024/kg_H2
2,12.0,6.960706,EUR_2024/kg_H2
3,13.0,6.628241,EUR_2024/kg_H2
4,14.0,6.343270,EUR_2024/kg_H2
...,...,...,...
85,95.0,3.184599,EUR_2024/kg_H2
86,96.0,3.178912,EUR_2024/kg_H2
87,97.0,3.173342,EUR_2024/kg_H2
88,98.0,3.167886,EUR_2024/kg_H2
